In [1]:
import os, re
import numpy as np
import scanpy as sc
from os.path import join
import pandas as pd

import sys
import scipy.io as sio
import scipy.sparse as sps
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from spamosaic.framework import SpaMosaic
import spamosaic.utils as utls
from spamosaic.preprocessing import RNA_preprocess, ADT_preprocess, Epigenome_preprocess, harmony

os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8' 

In [2]:
data_dir = '../../../data/processed/Embryo'

ad_sra_rna = sc.read_h5ad(join(data_dir, 'SRA/ad_rna.h5ad'))
ad_sra_atac = sc.read_h5ad(join(data_dir, 'SRA/ad_atac.h5ad'))

ad_mux2_27ac = sc.read_h5ad(join(data_dir, 'Mux2/ad_h3k27ac.h5ad'))
ad_mux2_27me = sc.read_h5ad(join(data_dir, 'Mux2/ad_h3k27me3.h5ad'))

ad_mux4_rna = sc.read_h5ad(join(data_dir, 'Mux4/ad_rna.h5ad'))
ad_mux4_atac = sc.read_h5ad(join(data_dir, 'Mux4/ad_atac.h5ad'))
ad_mux4_4me3 = sc.read_h5ad(join(data_dir, 'Mux4/ad_h3k4me3.h5ad'))
ad_mux4_27me = sc.read_h5ad(join(data_dir, 'Mux4/ad_h3k27me3.h5ad'))

ad_sat = sc.read_h5ad(join(data_dir, 'SAT/ad_atac.h5ad'))
ad_cut_27ac = sc.read_h5ad(join(data_dir, 'Cut-H3K27ac/ad_h3k27ac.h5ad'))
ad_cut_4me3 = sc.read_h5ad(join(data_dir, 'Cut-H3K4me3/ad_h3k4me3.h5ad'))
ad_cut_27me = sc.read_h5ad(join(data_dir, 'Cut-H3K27me3/ad_h3k27me3.h5ad'))

input_dict = { 
    'rna':      [ad_sra_rna, None,          ad_mux4_rna,  None,     None,       None,            None],
    'atac':     [ad_sra_atac,None,          ad_mux4_atac, ad_sat,   None,       None,            None],
    'h3k27me3': [None,       ad_mux2_27me,  ad_mux4_27me, None,     ad_cut_27me,None,            None],
    'h3k4me3':  [None,       None,          ad_mux4_4me3, None,     None,       ad_cut_4me3,     None],
    'h3k27ac':  [None,       ad_mux2_27ac,  None,         None,     None,       None,            ad_cut_27ac]
}

input_key = 'dimred_bc'
batch_key = 'Slice'

In [3]:
Epigenome_preprocess(input_dict['h3k27me3'], batch_corr=True, n_peak=50000, batch_key=batch_key, key=input_key, return_hvf=False)
Epigenome_preprocess(input_dict['h3k4me3'], batch_corr=True, n_peak=50000, batch_key=batch_key, key=input_key, return_hvf=False)
Epigenome_preprocess(input_dict['h3k27ac'], batch_corr=True, n_peak=50000, batch_key=batch_key, key=input_key, return_hvf=False)
Epigenome_preprocess(input_dict['atac'], batch_corr=True, n_peak=50000, batch_key=batch_key, key=input_key, return_hvf=False)
RNA_preprocess(input_dict['rna'], batch_corr=True, n_hvg=10000, batch_key=batch_key, key=input_key)

Use GPU mode.
	Initialization is completed.
	Completed 1 / 10 iteration(s).
	Completed 2 / 10 iteration(s).
Reach convergence after 2 iteration(s).
Use GPU mode.
	Initialization is completed.
	Completed 1 / 10 iteration(s).
	Completed 2 / 10 iteration(s).
Reach convergence after 2 iteration(s).
Use GPU mode.
	Initialization is completed.
	Completed 1 / 10 iteration(s).
	Completed 2 / 10 iteration(s).
Reach convergence after 2 iteration(s).
Use GPU mode.
	Initialization is completed.
	Completed 1 / 10 iteration(s).
	Completed 2 / 10 iteration(s).
Reach convergence after 2 iteration(s).
Use GPU mode.
	Initialization is completed.
	Completed 1 / 10 iteration(s).
	Completed 2 / 10 iteration(s).
	Completed 3 / 10 iteration(s).
	Completed 4 / 10 iteration(s).
Reach convergence after 4 iteration(s).


In [4]:
def stack(xl, key):
    xs, ns = [], []
    for adx in xl:
        if adx is not None:
            xs.append(adx.obsm[key])
            ns.append(adx.obs_names)
    df = pd.DataFrame(np.vstack(xs), index=np.hstack(ns))
    return df

for m1, m2 in zip(['rna', 'atac', 'h3k27me3', 'h3k4me3', 'h3k27ac'], ['RNA', 'ATAC', 'H3K27me3', 'H3K4me3', 'H3K27ac']):
    fig_dir = f'../../../results/embeddings/Leiden-{m2}/Embryo'
    os.makedirs(fig_dir, exist_ok=True)
    df = stack(input_dict[m1], input_key)
    df.to_csv(join(fig_dir, 'df_emb.csv'))